# Generador de clips — Wan 2.2 TI2V-5B (Kaggle Notebooks, gratis)

Este cuaderno genera los **clips de vídeo de un capítulo** con **Wan 2.2 TI2V-5B**,
un modelo abierto y gratuito de Alibaba, usando la **GPU gratuita de Kaggle**
(normalmente una NVIDIA T4 o P100).

**Cuota (informe de APIs gratis, 21 nov 2026): ~30 GPU-horas/semana**, sesiones de
hasta 12 h, sin tarjeta. Un clip de 5 s tarda unos 15–25 min en T4 (estimación,
sin medir en Kaggle): una semana de Kaggle da para ~70–120 clips de 5 s
(≈ 4–7 capítulos de 85 s), **siempre que la cuota semanal no se gaste en otra cosa**.

**Antes de empezar:**
1. Crea un **dataset** en Kaggle con la carpeta del capítulo: `jobs.json`
   (generado con `storyboard/preparar_jobs.py` del repo) y la subcarpeta
   `refs/` con las imágenes de referencia. Adjunta el dataset a este cuaderno
   (panel derecho → **Add data** → **Your Datasets**).
   - Alternativa sin dataset: sube `jobs.json` y `refs/` directamente a
     `/kaggle/working/` desde el panel de archivos del cuaderno.
2. Ve a **Settings → Accelerator** y elige **GPU T4 x2** (o el que haya).
3. Ejecuta todas las celdas en orden.

**Al terminar:** los clips quedan en `/kaggle/working/clips/`. Pulsa
**Save Version → Save & Run All** y descarga los archivos desde la pestaña
**Output** de la versión guardada. El cuaderno guarda cada clip en cuanto
termina, así que si la sesión muere a mitad, la siguiente sesión retoma
donde quedó (los clips hechos se saltan solos).


## 1. Comprobar la GPU

Kaggle tiene que habernos dado una GPU (en el nivel gratuito es una **NVIDIA T4**
o una **P100**). Si aquí dice "SIN GPU", ve a Settings → Accelerator, elige GPU
y vuelve a ejecutar esta celda.


In [ ]:
# Comprobamos que Kaggle nos dio una GPU (T4 o P100 en el nivel gratuito).
import time
import torch

INICIO_SESION = time.time()  # para el presupuesto de sesión (celda 5)

!nvidia-smi --query-gpu=name,memory.total --format=csv

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    GPU_NOMBRE = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU:", GPU_NOMBRE)
    print("VRAM total: %.1f GB" % VRAM_GB)
else:
    print("⚠️ SIN GPU: ve a Settings → Accelerator, elige GPU y re-ejecuta.")


## 2. Instalar las librerías

Instalamos `diffusers` (0.37 o superior, que ya incluye el soporte de Wan 2.2
TI2V-5B), `transformers`, `accelerate` y lo necesario para guardar los vídeos.
Kaggle ya trae `torch` y CUDA instalados. Tarda unos 3–5 minutos.


In [ ]:
# Instalamos las librerías necesarias (Kaggle ya trae torch y CUDA).
!pip install -q --upgrade "diffusers>=0.37" transformers accelerate ftfy imageio imageio-ffmpeg

import diffusers, transformers
print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("✅ Librerías listas.")


## 3. Cargar los jobs del capítulo

El cuaderno busca `jobs.json` en este orden:
1. En cualquier **dataset adjunto** (`/kaggle/input/*/jobs.json`).
2. En **`/kaggle/working/jobs.json`** (subido a mano al cuaderno).

Dentro de la misma carpeta debe estar la subcarpeta `refs/` con las imágenes
de referencia. Los clips terminados se guardan en `/kaggle/working/clips/` y el
registro en `/kaggle/working/registro.jsonl` (ambos se descargan con
"Save Version" → Output).


In [ ]:
# Cargamos los jobs del capítulo (desde el dataset o desde /kaggle/working).
import glob
import json
import os

# --- Rutas de Kaggle (normalmente no hay que cambiar nada) ---
CANDIDATOS_JOBS = sorted(glob.glob("/kaggle/input/*/jobs.json")) + [
    "/kaggle/working/jobs.json",
]
TRABAJO = "/kaggle/working"
CLIPS_DIR = os.path.join(TRABAJO, "clips")
REGISTRO = os.path.join(TRABAJO, "registro.jsonl")
os.makedirs(CLIPS_DIR, exist_ok=True)

JOBS_JSON = next((c for c in CANDIDATOS_JOBS if os.path.exists(c)), None)
if JOBS_JSON is None:
    raise FileNotFoundError(
        "No se encontró jobs.json. Adjunta el dataset del capítulo "
        "(Add data → Your Datasets) o sube jobs.json a /kaggle/working/ "
        "desde el panel de archivos, y vuelve a ejecutar esta celda.")

RAIZ_ENTRADA = os.path.dirname(JOBS_JSON)
REFS_DIR = os.path.join(RAIZ_ENTRADA, "refs")
if not os.path.isdir(REFS_DIR) and os.path.isdir(
        os.path.join(TRABAJO, "refs")):
    REFS_DIR = os.path.join(TRABAJO, "refs")

with open(JOBS_JSON, encoding="utf-8") as f:
    MANIFIESTO = json.load(f)
JOBS = MANIFIESTO["jobs"]
print(f"Capítulo: {MANIFIESTO.get('capitulo', '')} — {len(JOBS)} jobs cargados.")
print(f"jobs.json: {JOBS_JSON}")
print(f"Referencias: {REFS_DIR}")
print(f"Clips de salida: {CLIPS_DIR}")

# Estado actual: qué clips ya están hechos (se saltarán).
hechos = 0
for job in JOBS:
    ruta = os.path.join(CLIPS_DIR, job["clip_id"] + ".mp4")
    if os.path.exists(ruta) and os.path.getsize(ruta) > 10000:
        hechos += 1
    ref = os.path.join(REFS_DIR, job["referencia"])
    if job["referencia"] and not os.path.exists(ref):
        print(f"  ⚠️ {job['clip_id']}: falta la referencia {job['referencia']} en refs/")
print(f"Clips ya hechos: {hechos}/{len(JOBS)}. Faltan: {len(JOBS) - hechos}.")


## 4. Cargar el modelo Wan 2.2 TI2V-5B

Descargamos el modelo `Wan-AI/Wan2.2-TI2V-5B-Diffusers` de Hugging Face
(**~22 GB**, tarda 5–15 min la primera vez) y lo cargamos:

- El **VAE va en float32** y el resto en **bfloat16**, como indica la tarjeta
  oficial del modelo.
- `enable_model_cpu_offload()` mueve los pesos entre la GPU y la RAM para que
  el modelo quepa en los 16 GB de la T4 gratuita. Es más lento que en una
  tarjeta grande, pero funciona.

Nota: la P100 no tiene BF16 nativo tan rápido como la T4; si Kaggle da una P100
los clips pueden tardar más. El registro mide el tiempo real de cada clip.


In [ ]:
# Cargamos Wan 2.2 TI2V-5B (variante Diffusers).
import torch
from diffusers import WanImageToVideoPipeline, AutoencoderKLWan

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"

print("Cargando VAE (float32)…")
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", dtype=torch.float32)
print("Cargando pipeline Wan 2.2 TI2V-5B (descarga ~22 GB la primera vez)…")
pipe = WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=vae, dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
try:
    pipe.enable_vae_tiling()
    print("VAE tiling activado (ahorra VRAM al decodificar).")
except Exception as e:
    print("VAE tiling no disponible, se sigue sin él:", e)

print("✅ Modelo cargado y listo para generar.")


## 5. Generar los clips (reanudable)

El bucle genera los clips uno tras otro y **guarda cada uno en `/kaggle/working/clips/`
en cuanto termina** (la sesión puede morir en cualquier momento; nada se guarda solo
en el disco efímero de la sesión).

- **Cada clip tarda unos 15–25 min** en la T4 gratuita (estimación, sin medir en Kaggle).
- **Si un clip da falta de memoria en 720p**, se reintenta solo en 480p y
  queda registrado en `registro.jsonl`.
- **Presupuesto de sesión:** Kaggle corta las sesiones ~12 h. El bucle se
  detiene limpio antes de las 10,5 h; la siguiente sesión retoma donde quedó.
- **Medición:** cada clip registra su tiempo real, VRAM y GPU en
  `registro.jsonl`. Esos números son los que alimentan las estimaciones
  reales del orquestador (hoy son estimaciones sin medir).


In [ ]:
# Generamos los clips, uno tras otro, guardando cada uno en /kaggle/working.
import gc
import json
import os
import shutil
import time
import datetime

from PIL import Image
from diffusers.utils import export_to_video

NEGATIVE_PROMPT = (
    "Bright tones, overexposed, static, blurred details, subtitles, style, works, "
    "paintings, images, static, overall gray, worst quality, low quality, "
    "JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, "
    "poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, "
    "still picture, messy background, three legs, text, words, letters, captions, "
    "on-screen text, watermark"
)

FPS = MANIFIESTO.get("fps", 24)
LIMITE_SESION_H = 10.5       # Kaggle corta ~12 h: parar limpio antes
ESTIMACION_MIN_CLIP = 20.0   # se ajusta con los tiempos reales medidos
RESOLUCION_FALLBACK = (480, 832)  # vertical 480p, la verificada en T4

os.makedirs("clips-tmp", exist_ok=True)


def preparar_referencia(ruta, ancho, alto):
    """Recorta la referencia a la proporción del vídeo (sin deformar) y
    la ajusta a la resolución del job."""
    img = Image.open(ruta).convert("RGB")
    objetivo = ancho / alto
    w, h = img.size
    if w / h > objetivo:  # sobra ancho: recortar lados
        nuevo_w = int(h * objetivo)
        x0 = (w - nuevo_w) // 2
        img = img.crop((x0, 0, x0 + nuevo_w, h))
    else:  # sobra alto: recortar arriba/abajo
        nuevo_h = int(w / objetivo)
        y0 = (h - nuevo_h) // 2
        img = img.crop((0, y0, w, y0 + nuevo_h))
    return img.resize((ancho, alto), Image.LANCZOS)


def registrar(entrada):
    with open(REGISTRO, "a", encoding="utf-8") as f:
        f.write(json.dumps(entrada, ensure_ascii=False) + "\n")


def generar_un_clip(job, ancho, alto):
    """Genera un clip con el pipeline. Devuelve la ruta local."""
    ref = preparar_referencia(os.path.join(REFS_DIR, job["referencia"]), ancho, alto)
    torch.cuda.reset_peak_memory_stats()
    generator = torch.Generator(device="cpu").manual_seed(job["seed"])
    output = pipe(
        image=ref,
        prompt=job["prompt_video"],
        negative_prompt=NEGATIVE_PROMPT,
        height=alto,
        width=ancho,
        num_frames=job["num_frames"],
        num_inference_steps=job["pasos"],
        guidance_scale=job["guidance"],
        generator=generator,
    ).frames[0]
    ruta_tmp = f"clips-tmp/{job['clip_id']}.mp4"
    export_to_video(output, ruta_tmp, fps=FPS)
    vram_gb = torch.cuda.max_memory_allocated() / 1e9
    del output
    gc.collect()
    torch.cuda.empty_cache()
    return ruta_tmp, vram_gb


tiempos_reales = []
for i, job in enumerate(JOBS, start=1):
    destino = os.path.join(CLIPS_DIR, job["clip_id"] + ".mp4")
    if os.path.exists(destino) and os.path.getsize(destino) > 10000:
        print(f"Clip {job['clip_id']} ya existe, se salta.")
        continue

    # Presupuesto de sesión: ¿cabe otro clip antes del corte?
    transcurrido_h = (time.time() - INICIO_SESION) / 3600
    estimacion = (sum(tiempos_reales) / len(tiempos_reales)
                  if tiempos_reales else ESTIMACION_MIN_CLIP)
    if transcurrido_h + (estimacion * 1.2) / 60 > LIMITE_SESION_H:
        print(f"⏸️  Se para limpio: han pasado {transcurrido_h:.1f} h de sesión y "
              f"otro clip (~{estimacion:.0f} min) no cabe antes del límite "
              f"de {LIMITE_SESION_H} h. Ejecuta otra sesión para continuar.")
        break

    ancho, alto = job["resolucion"]
    print(f"\n=== {job['clip_id']} ({i}/{len(JOBS)}) — {ancho}x{alto}, "
          f"{job['num_frames']} fotogramas, seed {job['seed']} ===", flush=True)
    t0 = time.time()
    inicio_iso = datetime.datetime.now(datetime.timezone.utc).isoformat()
    try:
        ruta_tmp, vram = generar_un_clip(job, ancho, alto)
        res_final = [ancho, alto]
    except RuntimeError as e:
        if "out of memory" in str(e).lower() and (ancho, alto) != RESOLUCION_FALLBACK:
            print(f"  ⚠️ Falta de memoria en {ancho}x{alto}: reintento en "
                  f"{RESOLUCION_FALLBACK[0]}x{RESOLUCION_FALLBACK[1]}.", flush=True)
            torch.cuda.empty_cache()
            gc.collect()
            ancho, alto = RESOLUCION_FALLBACK
            ruta_tmp, vram = generar_un_clip(job, ancho, alto)
            res_final = [ancho, alto]
        else:
            raise
    minutos = (time.time() - t0) / 60
    tiempos_reales.append(minutos)
    shutil.copy2(ruta_tmp, destino)  # guardado inmediato en /kaggle/working
    os.remove(ruta_tmp)
    print(f"✅ {job['clip_id']} listo en {minutos:.1f} min -> {destino} "
          f"(VRAM máx. {vram:.1f} GB)", flush=True)
    registrar({
        "clip_id": job["clip_id"],
        "estado": "hecho",
        "inicio_utc": inicio_iso,
        "duracion_min": round(minutos, 2),
        "resolucion": res_final,
        "pasos": job["pasos"],
        "guidance": job["guidance"],
        "seed": job["seed"],
        "vram_max_gb": round(vram, 2),
        "gpu": GPU_NOMBRE,
        "plataforma": "kaggle",
    })

print("\n--- Resumen de la sesión ---")
if tiempos_reales:
    print(f"Clips generados en esta sesión: {len(tiempos_reales)}")
    print(f"Tiempo medio por clip: {sum(tiempos_reales) / len(tiempos_reales):.1f} min "
          f"(mín. {min(tiempos_reales):.1f}, máx. {max(tiempos_reales):.1f})")
print(f"Registro completo en: {REGISTRO}")
print("Descarga los clips con Save Version → Output.")


## 6. Unir los clips y empaquetar el ZIP

Verificamos que estén todos los clips, los unimos en un solo vídeo (sin
audio: la narración se añade después con Piper TTS en la laptop) y empaquetamos
todo — los clips sueltos + el vídeo unido — en un ZIP en `/kaggle/working/`.
Descarga el ZIP desde **Save Version → Output**.


In [ ]:
# Verificamos los clips, los unimos (sin audio) y empaquetamos un ZIP.
import glob
import os
import subprocess
import zipfile

clips = sorted(glob.glob(os.path.join(CLIPS_DIR, "clip*.mp4")))
print(f"Clips encontrados: {len(clips)}/{len(JOBS)}")
for c in clips:
    print(f"  {os.path.basename(c)} — {os.path.getsize(c) / 1e6:.1f} MB")
assert len(clips) == len(JOBS), "Faltan clips: vuelve a ejecutar la celda de generación."

video_unido = os.path.join(TRABAJO, "capitulo-sin-audio.mp4")
lista = os.path.join(TRABAJO, "lista-clips.txt")
with open(lista, "w") as f:
    for c in clips:
        f.write(f"file '{c}'\n")
subprocess.run([
    "ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", lista,
    "-c:v", "libx264", "-crf", "20", "-pix_fmt", "yuv420p",
    "-movflags", "+faststart", video_unido,
], check=True, capture_output=True)
print(f"Vídeo unido: {video_unido} — {os.path.getsize(video_unido) / 1e6:.1f} MB")

zip_path = os.path.join(TRABAJO, "clips.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for c in clips:
        z.write(c, os.path.basename(c))
    z.write(video_unido, os.path.basename(video_unido))
print(f"ZIP creado: {zip_path} — {os.path.getsize(zip_path) / 1e6:.1f} MB")
print("Descárgalo con Save Version → Output.")


In [ ]:
# Vemos el resultado.
from IPython.display import display, Video

display(Video(video_unido, width=360, height=640))
print("Para descargar los clips: Save Version → Save & Run All → pestaña Output.")


## 7. Qué sigue y solución de problemas

**Qué sigue:**
1. Descarga los clips (o el ZIP) desde **Save Version → Output** a la laptop.
2. Recógelos con el orquestador: `python -m orquestador.cli recolectar
   --trabajo cap-01/ --clips clips/` (verifica duración y que no estén corruptos).
3. Genera la narración con Piper: `python voz/piper_tts.py --guion guion.json
   --salida wavs/` (o deja que `ensamblar` lo haga).
4. Ensambla el capítulo: `python -m orquestador.cli ensamblar --guion guion.json
   --trabajo cap-01/ --clips clips/ --wavs wavs/ --salida capitulo.mp4`.
5. El QC corre dentro de `ensamblar`. Un capítulo que no pasa NO se publica.

**Si algo falla:**
- **`CUDA out of memory` en 720p:** el cuaderno reintenta solo en 480p.
- **La sesión muere a mitad:** vuelve a ejecutar desde la celda 1; los clips
  hechos se saltan solos (si descargaste los clips, súbelos de nuevo a
  `/kaggle/working/clips/` o el cuaderno los regenerará).
- **Kaggle dice que se acabó la cuota semanal de GPU:** espera al lunes
  siguiente (la cuota es de ~30 h/semana) o usa Colab para los clips que falten.
- **La P100 tarda más que la T4:** es normal; el registro mide el tiempo real.
  Si un clip tarda más de 40 min, baja los pasos a 30 en los jobs.
